# Lab 01: Coin Flip Simulation

*SF250 Probability Theory and Statistics. Written in C, run from this notebook.*


# Coin Flip Simulation

Flip a (possibly biased) coin N times in C, then show how the results behave:
the running proportion of heads settles toward the true probability, the
number of heads per group follows a Binomial distribution, and a simple
z-test decides whether the coin looks fair.

## Build and run

```
make
./coin_flip [N] [p_head] [seed]
```

| Argument | Meaning | Default |
|---|---|---|
| `N` | number of flips (must be >= 10) | 1000 |
| `p_head` | probability of heads, 0 < p < 1 | 0.5 |
| `seed` | random seed (same seed gives the same run) | current time |

## How it works

1. **Simulation.** Each flip draws a uniform random number and counts a head
   when it is below `p`. Each flip is a Bernoulli(p) trial.
2. **Convergence plot.** At 20 checkpoints, spaced on a log scale, it prints
   the running proportion `p_hat = heads / n`. On the plot `*` is `p_hat` and
   `|` is the true `p`. For small N some checkpoints repeat the same `n`.
3. **Binomial histogram.** The flips are cut into groups of 10. For each
   group it counts the heads (0 to 10), then compares the observed number of
   groups with the Binomial(10, p) expectation `groups * C(10,k) p^k (1-p)^(10-k)`.
4. **Test.** Under H0 (the coin really has probability `p`) the number of
   heads has mean `np` and standard deviation `sqrt(np(1-p))`. It prints
   `z = (heads - np) / SD` and rejects H0 when `|z| >= 1.96` (alpha = 0.05).

## Makefile shortcuts

```
make                         build ./coin_flip
make run                     build and run the example: ./coin_flip 1000 0.5 42
make run ARGS="5000 0.7 1"   run with your own arguments
make clean                   remove the program
make re                      clean, then build again
```

## Examples

```
./coin_flip                   # 1000 flips, fair coin
./coin_flip 100000 0.5 42     # 100,000 flips, reproducible
./coin_flip 5000 0.7 1       # biased coin with P(H) = 0.7
```

Output of `./coin_flip 1000 0.5 7` (trimmed):

```
== Coin flip simulation: N=1000, P(H)=0.500, seed=7 ==

Convergence of running proportion (Law of Large Numbers)
        n   p_hat  0                                                 1
     ...
     1000  0.4840                          *|

Heads per group of 10 flips: observed (#) vs binomial expected
  k     obs  expected
  4      25      20.5 #####################################
  5      27      24.6 ########################################
  6      16      20.5 ########################

Heads=484  Tails=516  p_hat=0.4840
Expected heads=500.0  SD=15.81  z=-1.012
H0: P(H)=0.500 at alpha=0.05 -> fail to reject
```

Things to try:
- Raise N (10, 1000, 100000) and watch `*` move onto `|`.
- Run the same N with different seeds; the histogram bars wobble around the
  expected counts.
- The test checks the data against the `p_head` you passed, so it rejects
  only about 5% of runs (that is what alpha = 0.05 means).
- "Fail to reject" does not prove the coin is fair, it only means the data
  are not unusual enough under H0.


## Slide references

ชื่อย่อสไลด์ (เลขหลังชื่อคือเลขหน้า "Slide" มุมขวาล่าง):

| ย่อ | ชุดสไลด์ |
|---|---|
| **PF** | Probability Fundamentals |
| **DD** | Discrete Probability Distributions |
| **CD** | Continuous Distributions |
| **MGF** | Moment Generating Functions |
| **CM** | Combining and Manipulating Distributions |
| **TS** | Testing and Statistical Significance |

## 1. `coin_flip`: จำลองการโยนเหรียญ

**คืออะไร:** โยนเหรียญ N ครั้ง (หัวออกด้วยความน่าจะเป็น p) แล้วสรุปผลเป็นกราฟและสถิติ

**ทำงานยังไง:**
1. โยนทีละครั้ง นับหัวสะสม แล้วพิมพ์ p̂ = heads/n ที่ 20 จุด เพื่อดูว่าค่าเข้าใกล้ p
2. แบ่งการโยนเป็นกลุ่มละ 10 ครั้ง นับจำนวนหัวต่อกลุ่ม แล้วเทียบกับ Binomial(10, p)
3. คำนวณ z = (heads − np) / √(np(1−p)) แล้วตัดสิน H0 ที่ |z| ≥ 1.96

| ส่วนของโปรแกรม | อ้างอิงสไลด์ |
|---|---|
| ความน่าจะเป็น = ความถี่ระยะยาว (กราฟ p̂ ลู่เข้า p) | PF 5 |
| เหรียญหนึ่งครั้ง = Bernoulli, mean = p, var = p(1−p) | DD 4-5 |
| จำนวนหัวต่อ 10 ครั้ง = Binomial, pmf = nCr·p^r·(1−p)^(n−r) | DD 7 (กราฟ DD 10) |
| nCr คืออะไร | DD 8 |
| E[X] = np และ Var = np(1−p) | DD 11-12 |
| ประมาณ Binomial ด้วย Gaussian แล้วแปลงเป็นค่ามาตรฐาน z | TS 5, CD 9 |
| สมมติฐานว่าง H0, "ค่าที่สังเกตหรือสุดโต่งกว่านั้น" | TS 3-4, 7 |
| ทดสอบสองด้านกับเหรียญ (เหรียญยุติธรรมไหม) | TS 8 |
| ไม่ reject H0 ไม่ได้แปลว่า H0 จริง | TS 16 |
| ช่วง ±2σ ≈ 95% (ที่มาของเกณฑ์ |z| ≥ 1.96) | CD 11 |

ทุกส่วนอยู่ในสไลด์


## Running C in a notebook

This notebook uses the normal Python kernel. A cell that starts with `%%writefile` saves a C source file, and a cell that starts with `!` runs a shell command (`make`, then the compiled program). It needs `gcc` and `make`, which Google Colab already has. On a local Jupyter, install them first (for example `sudo apt install build-essential`). The C code is the same code as in the repository, written in 42 Norm style.


In [1]:
!mkdir -p coin_flip

In [2]:
!gcc --version | head -1

gcc (GCC) 16.2.1 20261029 (Red Hat 16.2.1-3)


## Source code

One cell per file. Run them all to recreate the project.

In [3]:
%%writefile coin_flip/coin_flip.h
#ifndef COIN_FLIP_H
# define COIN_FLIP_H

# include <math.h>
# include <stdio.h>
# include <stdlib.h>
# include <string.h>
# include <time.h>

# define K 10
# define NC 20
# define W 50

typedef struct s_coin
{
	long			n;
	long			heads;
	long			groups;
	long			hist[K + 1];
	double			p;
	unsigned int	seed;
}	t_coin;

void	simulate(t_coin *c);
void	print_hist(t_coin *c);
void	print_stats(t_coin *c);

#endif


Writing coin_flip/coin_flip.h


In [4]:
%%writefile coin_flip/coin_flip.c
#include "coin_flip.h"

static int	parse(t_coin *c, int ac, char **av)
{
	c->n = 1000;
	c->p = 0.5;
	c->seed = (unsigned int)time(NULL);
	if (ac > 1)
		c->n = atol(av[1]);
	if (ac > 2)
		c->p = atof(av[2]);
	if (ac > 3)
		c->seed = (unsigned int)atol(av[3]);
	if (c->n < K || c->p <= 0 || c->p >= 1)
	{
		fprintf(stderr, "usage: %s [N>=%d] [0<p<1] [seed]\n", av[0], K);
		return (1);
	}
	return (0);
}

static void	print_head(t_coin *c)
{
	printf("== Coin flip simulation: N=%ld, P(H)=%.3f, seed=%u ==\n\n", c->n,
		c->p, c->seed);
	printf("Convergence of running proportion (Law of Large Numbers)\n");
	printf("%9s %7s  0%*s1\n", "n", "p_hat", W - 1, "");
}

int	main(int ac, char **av)
{
	t_coin	c;

	memset(&c, 0, sizeof(c));
	if (parse(&c, ac, av))
		return (1);
	srand(c.seed);
	print_head(&c);
	simulate(&c);
	print_hist(&c);
	print_stats(&c);
	return (0);
}


Writing coin_flip/coin_flip.c


In [5]:
%%writefile coin_flip/coin_flip_sim.c
#include "coin_flip.h"

static void	row(t_coin *c, long i)
{
	double	ph;
	int		col;
	int		mid;
	int		x;

	ph = (double)c->heads / i;
	col = (int)(ph * (W - 1) + 0.5);
	mid = (int)(c->p * (W - 1) + 0.5);
	printf("%9ld %7.4f  ", i, ph);
	x = -1;
	while (++x < W)
	{
		if (x == col)
			putchar('*');
		else if (x == mid)
			putchar('|');
		else
			putchar(' ');
	}
	putchar('\n');
}

static int	due(t_coin *c, long i, int ci)
{
	return (ci <= NC && i >= (long)ceil(pow((double)c->n, (double)ci / NC)));
}

static void	tally(t_coin *c, long i, int *gh)
{
	if (rand() / (RAND_MAX + 1.0) < c->p)
	{
		c->heads++;
		(*gh)++;
	}
	if (i % K == 0)
	{
		c->hist[*gh]++;
		c->groups++;
		*gh = 0;
	}
}

void	simulate(t_coin *c)
{
	long	i;
	int		gh;
	int		ci;

	gh = 0;
	ci = 1;
	i = 0;
	while (++i <= c->n)
	{
		tally(c, i, &gh);
		while (due(c, i, ci))
		{
			row(c, i);
			ci++;
		}
	}
}


Writing coin_flip/coin_flip_sim.c


In [6]:
%%writefile coin_flip/coin_flip_print.c
#include "coin_flip.h"

static void	bar(int n)
{
	while (n-- > 0)
		putchar('#');
	putchar('\n');
}

static double	expected(t_coin *c, int k)
{
	return (c->groups * tgamma(K + 1) / (tgamma(k + 1) * tgamma(K - k + 1))
		* pow(c->p, k) * pow(1 - c->p, K - k));
}

void	print_hist(t_coin *c)
{
	int		k;
	double	maxc;

	printf("\nHeads per group of %d flips: ", K);
	printf("observed (#) vs binomial expected\n");
	printf("%3s %7s %9s\n", "k", "obs", "expected");
	maxc = 1;
	k = -1;
	while (++k <= K)
		if (c->hist[k] > maxc)
			maxc = c->hist[k];
	k = -1;
	while (++k <= K)
	{
		printf("%3d %7ld %9.1f ", k, c->hist[k], expected(c, k));
		bar((int)(40.0 * c->hist[k] / maxc + 0.5));
	}
}

void	print_stats(t_coin *c)
{
	double	ph;
	double	sd;
	double	z;

	ph = (double)c->heads / c->n;
	sd = sqrt(c->n * c->p * (1 - c->p));
	z = (c->heads - c->n * c->p) / sd;
	printf("\nHeads=%ld  Tails=%ld  p_hat=%.4f\n", c->heads, c->n - c->heads,
		ph);
	printf("Expected heads=%.1f  SD=%.2f  z=%.3f\n", c->n * c->p, sd, z);
	printf("H0: P(H)=%.3f at alpha=0.05 -> ", c->p);
	if (fabs(z) < 1.96)
		printf("fail to reject\n");
	else
		printf("REJECT\n");
}


Writing coin_flip/coin_flip_print.c


In [7]:
%%writefile coin_flip/Makefile
NAME = coin_flip
SRCS = coin_flip.c coin_flip_sim.c coin_flip_print.c
CFLAGS = -Wall -Wextra -O2
ARGS = 1000 0.5 42

all: $(NAME)

$(NAME): $(SRCS) coin_flip.h
	$(CC) $(CFLAGS) $(SRCS) -o $(NAME) -lm

# make run               runs the example above
# make run ARGS="..."    runs with your own arguments
run: $(NAME)
	./$(NAME) $(ARGS)

clean:
	rm -f $(NAME)

re: clean all

.PHONY: all run clean re


Writing coin_flip/Makefile


## Build

`make` compiles every `.c` file into one program.

In [8]:
!cd coin_flip && make

cc -Wall -Wextra -O2 coin_flip.c coin_flip_sim.c coin_flip_print.c -o coin_flip -lm


## Run it

A fair coin, 1000 flips (seed 7 makes the run repeatable):

In [9]:
!cd coin_flip && ./coin_flip 1000 0.5 7

== Coin flip simulation: N=1000, P(H)=0.500, seed=7 ==

Convergence of running proportion (Law of Large Numbers)
        n   p_hat  0                                                 1
        2  0.5000                           *                        
        2  0.5000                           *                        
        3  0.3333                  *        |                        
        4  0.5000                           *                        
        6  0.5000                           *                        
        8  0.5000                           *                        
       12  0.5000                           *                        
       16  0.4375                       *   |                        
       23  0.5217                           |*                       
       32  0.5625                           |  *                     
       45  0.4444                        *  |                        
       64  0.3906                     *     | 

More flips: the running proportion locks onto 0.5.

In [10]:
!cd coin_flip && ./coin_flip 100000 0.5 42

== Coin flip simulation: N=100000, P(H)=0.500, seed=42 ==

Convergence of running proportion (Law of Large Numbers)
        n   p_hat  0                                                 1
        2  1.0000                           |                       *
        4  0.7500                           |           *            
        6  0.8333                           |               *        
       11  0.7273                           |          *             
       18  0.6667                           |       *                
       32  0.4688                         * |                        
       57  0.4737                         * |                        
      101  0.4653                         * |                        
      178  0.4663                         * |                        
      317  0.4826                          *|                        
      563  0.4956                          *|                        
     1000  0.5170                          

A biased coin with P(heads) = 0.7:

In [11]:
!cd coin_flip && ./coin_flip 5000 0.7 1

== Coin flip simulation: N=5000, P(H)=0.700, seed=1 ==

Convergence of running proportion (Law of Large Numbers)
        n   p_hat  0                                                 1
        2  0.5000                           *        |               
        3  0.3333                  *                 |               
        4  0.2500              *                     |               
        6  0.3333                  *                 |               
        9  0.4444                        *           |               
       13  0.6154                                *   |               
       20  0.6000                               *    |               
       31  0.6774                                   *|               
       47  0.6596                                  * |               
       71  0.6761                                   *|               
      109  0.6514                                  * |               
      166  0.6687                             

## Try it yourself

Change the numbers in a command above and re-run the cell. The arguments are described in the usage section at the top of this notebook.